In [ ]:
!sudo apt-get install libsndfile1
!pip install soundfile
!pip install torchcodec

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
import os
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
from io import BytesIO
from transformers import Wav2Vec2Model, Wav2Vec2Processor #for Wav2Vec
import torch
import torchaudio
import torchaudio.functional as AF
import torchaudio.transforms as T


from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Supported formats:
valid_extensions = ['.wav', '.flac'] #flac for ASVspoof and wav for In-the-Wild

Baseline: Log-Mel Spectrogram


Example:

<img src="https://drive.google.com/uc?export=view&id=14nmr4WBVQoo_1RY2Zlg59qk5iJwsJ89o" width="300">

In [ ]:
# Input and output directories:
input_folder = "/content/drive/MyDrive/Dataset_2025/Audio/test/ASV19LA/bonafide/"   #Update it based on the dataset/subset/class that you are converting
output_folder = "/content/drive/My Drive/Dataset_2025/Images/test/ASV19LA/bonafide/"#Update it based on the dataset/subset/class that you are converting
os.makedirs(output_folder, exist_ok=True)

In [ ]:
# Function to save a spectrogram image:
def save_log_mel_spectrogram(audio_path, save_path, sr=16000, n_fft=1024,
                              hop_length=256, n_mels=128, duration=4.0,
                              target_size=(224, 224)): #The data preperation, sampling rate = 16 KHz, duration = 4 seconds, and tail-padding
    try:
        y, _ = librosa.load(audio_path, sr=sr)

        # Trim or tile-pad to exactly 4 seconds:
        required_length = int(sr * duration)
        if len(y) < required_length:
            n_repeat = int(np.ceil(required_length / len(y)))
            y = np.tile(y, n_repeat)[:required_length]
        else:
            y = y[:required_length]

        # Generate log-mel spectrogram:
        S = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=n_fft,
                                           hop_length=hop_length, n_mels=n_mels)
        S_dB = librosa.power_to_db(S, ref=np.max, top_db=80)

        # Plot and save to memory:
        fig = plt.figure(figsize=(2.24, 2.24), dpi=100)
        ax = fig.add_axes([0, 0, 1, 1])
        ax.axis('off')
        img_buffer = BytesIO()
        librosa.display.specshow(S_dB, sr=sr, hop_length=hop_length, cmap='viridis', ax=ax)
        ax.set_ylim(0, S_dB.shape[0])
        ax.set_aspect('auto')
        plt.savefig(img_buffer, format='png', dpi=100)
        plt.close(fig)
        img_buffer.seek(0)

        # Open from buffer and resize:
        img = Image.open(img_buffer).convert("RGB")
        img = img.resize(target_size, Image.BICUBIC)
        img.save(save_path)
        img_buffer.close()

        #print(f"Saved: {save_path}")

    except Exception as e:
        print(f"Error: {os.path.basename(audio_path)}, {e}")
        with open("failed_files.txt", "a") as f:
            f.write(f"{audio_path} - {e}\n")


# Function to process files:
def process_file(filename):
    if not any(filename.lower().endswith(ext) for ext in valid_extensions):
        return
    name = os.path.splitext(filename)[0]
    output_path = os.path.join(output_folder, name + ".png")
    if os.path.exists(output_path):
        return
    try:
        audio_path = os.path.join(input_folder, filename)
        save_log_mel_spectrogram(audio_path, output_path)
    except Exception as e:
        print(f"Error: {filename}, {e}")

In [ ]:
# Batch processing with progress bar and multithreading to see the progress:
with ThreadPoolExecutor(max_workers=1) as executor:
    list(tqdm(executor.map(process_file, os.listdir(input_folder)),
              total=len(os.listdir(input_folder)),
              desc="Converting spectrograms"))

Proposed Variant I: Dual-Representation Stacked Fusion

<img src="https://drive.google.com/uc?export=view&id=1vSGQ3H-wUK3eeATcAoT4gotXY38oNb0z" width="300">


In [ ]:
# Input and output directories
input_folder = "/content/drive/MyDrive/Dataset_2025/Audio/test/ASV19LA/bonafide/"     #Update it based on the dataset/subset/class that you are converting
output_folder = "/content/drive/My Drive/Dataset_2025/ImagesV2/test/ASV19LA/bonafide/"#Update it based on the dataset/subset/class that you are converting

In [ ]:
def save_fused_wav2vec_mel_image(
    audio_path, save_path, device, model, processor,
    sr=16000, n_fft=1024, hop_length=256, n_mels=128,
    duration=4.0, target_size=(224,224)
):
    try:
        # Load and pad/tile audio:
        y, _ = librosa.load(audio_path, sr=sr)
        required_length = int(sr * duration)
        if len(y) < required_length:
            n_repeat = int(np.ceil(required_length / len(y)))
            y = np.tile(y, n_repeat)[:required_length]
        else:
            y = y[:required_length]

        # Mel-spectrogram:
        S = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=n_fft,
                                           hop_length=hop_length, n_mels=n_mels)
        S_dB = librosa.power_to_db(S, ref=np.max, top_db=80)
        mel_img_buffer = BytesIO()
        fig = plt.figure(figsize=(2.24, 2.24), dpi=100)
        ax = fig.add_axes([0, 0, 1, 1])
        ax.axis('off')
        librosa.display.specshow(S_dB, sr=sr, hop_length=hop_length, cmap='gray', ax=ax)
        ax.set_ylim(0, S_dB.shape[0])
        ax.set_aspect('auto')
        plt.savefig(mel_img_buffer, format='png', dpi=100)
        plt.close(fig)
        mel_img_buffer.seek(0)
        mel_img = Image.open(mel_img_buffer).convert("L").resize(target_size, Image.BICUBIC)
        mel_img_buffer.close()
        mel_array = np.array(mel_img)

        # Wav2vec2.0 embedding:
        waveform, tor_sr = torchaudio.load(audio_path)
        waveform = waveform.squeeze(0)
        if tor_sr != sr:
            waveform = torchaudio.transforms.Resample(orig_freq=tor_sr, new_freq=sr)(waveform)
        waveform = waveform.cpu().numpy()
        # Crop/tile:
        if waveform.shape[0] < required_length:
            n_repeat = int(np.ceil(required_length / waveform.shape[0]))
            waveform = np.tile(waveform, n_repeat)[:required_length]
        else:
            waveform = waveform[:required_length]
        waveform = torch.tensor(waveform)
        # To model:
        inputs = processor(waveform, sampling_rate=sr, return_tensors="pt").input_values.to(device)
        with torch.no_grad():
            feat = model(inputs).last_hidden_state.squeeze(0).cpu().numpy()  # (frames, 768)
        # Interpolate to (224, 224):
        from scipy.ndimage import zoom
        zoom_h = target_size[0] / feat.shape[0]
        zoom_w = target_size[1] / feat.shape[1]
        wav2vec_img = zoom((feat - feat.min()) / (feat.max() - feat.min() + 1e-8), (zoom_h, zoom_w), order=3)
        wav2vec_img_uint8 = (wav2vec_img * 255).astype(np.uint8)
        wav2vec_img_pil = Image.fromarray(wav2vec_img_uint8, mode='L')

        # Fuse vertically: [wav2vec on top, mel on bottom]
        fused = Image.new("L", (target_size[0], target_size[1]*2))
        fused.paste(wav2vec_img_pil, (0,0))
        fused.paste(mel_img, (0,target_size[1]))
        fused = fused.resize(target_size, resample=Image.BICUBIC)
        fused.save(save_path, format="PNG", quality=95)

    except Exception as e:
        print(f"Error: {os.path.basename(audio_path)}, {e}")
        with open("failed_files.txt", "a") as f:
            f.write(f"{audio_path} - {e}\n")


# Wav2vec2.0 embedding:
wav2vec_model_name = "facebook/wav2vec2-base"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
processor = Wav2Vec2Processor.from_pretrained(wav2vec_model_name)
model = Wav2Vec2Model.from_pretrained(wav2vec_model_name).to(device).eval()

# Function to process files:
def process_file(filename):
    if not any(filename.lower().endswith(ext) for ext in valid_extensions):
        return
    name = os.path.splitext(filename)[0]
    output_path = os.path.join(output_folder, name + ".png")
    if os.path.exists(output_path):
        return
    try:
        audio_path = os.path.join(input_folder, filename)
        save_fused_wav2vec_mel_image(audio_path, output_path, device, model, processor)
    except Exception as e:
        print(f"Error: {filename}, {e}")

In [ ]:
# Batch processing with progress bar and multithreading to see the progress:
with ThreadPoolExecutor(max_workers=1) as executor:
    list(tqdm(executor.map(process_file, os.listdir(input_folder)),
              total=len(os.listdir(input_folder)),
              desc="Converting spectrograms"))

Proposed Variant II: Tri-Channel Spectro-Contextual Fusion

<img src="https://drive.google.com/uc?export=view&id=1LL1HCepDRGypfbWiIpqllObUB5xoCtt8" width="300">


In [ ]:
# Input and output directories:
input_folder = "/content/drive/MyDrive/Dataset_2025/Audio/test/ASV19LA/bonafide/"          #Update it based on the dataset/subset/class that you are converting
output_folder = "/content/drive/My Drive/Dataset_2025/Images_Fusion/test/ASV19LA/bonafide/"#Update it based on the dataset/subset/class that you are converting

In [ ]:
#Parameters: (for the data preperation and input representaion)
SR = 16000
DURATION = 4.0
N_MELS = 128
N_FFT = 1024
HOP = 160
FMIN, FMAX = 20.0, 7600.0
TOP_DB = 80.0
OUT_H = 224
OUT_W = 224
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Funcitons:
def load_pad_crop(path, sr=SR, duration=DURATION):
    wav, in_sr = torchaudio.load(path)
    if in_sr != sr:
        wav = AF.resample(wav, in_sr, sr)
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    target = int(sr * duration)
    if wav.shape[-1] < target:
        wav = torch.nn.functional.pad(wav, (0, target - wav.shape[-1]))
    elif wav.shape[-1] > target:
        wav = wav[..., :target]
    wav = wav / wav.abs().max().clamp(min=1e-6)
    return wav

def robust_scale_0_255(x, p_lo=1.0, p_hi=99.0):
    x_np = x.detach().cpu().float().numpy()
    lo, hi = np.percentile(x_np, [p_lo, p_hi])
    if hi <= lo: hi = lo + 1e-6
    y = np.clip((x_np - lo) / (hi - lo), 0.0, 1.0)
    return (y * 255.0 + 0.5).astype(np.uint8)

def pil_resize_uint8(arr, h=OUT_H, w=OUT_W):
    im = Image.fromarray(arr)
    im = im.resize((w, h), resample=Image.BILINEAR)
    return np.array(im)

def save_rgb_resized(mel, delta, wmap, out_png):
    R = robust_scale_0_255(mel)
    G = robust_scale_0_255(delta)
    B = robust_scale_0_255(wmap)
    rgb = np.stack([R, G, B], axis=-1)
    rgb_resized = pil_resize_uint8(rgb, OUT_H, OUT_W)
    Image.fromarray(rgb_resized).save(out_png)

# Prepare mel and wav2vec:
mel_tf = T.MelSpectrogram(sample_rate=SR, n_fft=N_FFT, hop_length=HOP,
                          n_mels=N_MELS, f_min=FMIN, f_max=FMAX, power=2.0)
to_db = T.AmplitudeToDB(top_db=TOP_DB)
w2v = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base").to(DEVICE)
w2v.eval()

# Loop through files:
files = [f for f in os.listdir(input_folder) if f.lower().endswith(('.flac', '.wav'))]
print(f"Found {len(files)} audio files.")

#Loop through files to create Variant II:
for fname in tqdm(files):
    in_path = os.path.join(input_folder, fname)
    out_path = os.path.join(output_folder, os.path.splitext(fname)[0] + '.png')

    try:
        wav = load_pad_crop(in_path)

        # mel + delta:
        mel = to_db(mel_tf(wav))
        mel = (mel - mel.mean()) / mel.std().clamp(min=1e-6)
        delta = AF.compute_deltas(mel.squeeze(0)).unsqueeze(0)
        delta = (delta - delta.mean()) / delta.std().clamp(min=1e-6)

        # wav2vec:
        w2v_in = wav.squeeze(0).unsqueeze(0).to(DEVICE).float()
        with torch.no_grad():
            h = w2v(input_values=w2v_in).last_hidden_state  # (1, Tw, 768)
        h = h.transpose(1, 2)  # (1, 768, Tw)
        Tm = mel.shape[-1]
        h_aligned = torch.nn.functional.interpolate(h, size=Tm, mode='linear', align_corners=False)

        # projection: [768 -> N_MELS]
        proj = torch.zeros((N_MELS, 768), dtype=h_aligned.dtype, device=h_aligned.device)
        group = 768 // N_MELS if 768 >= N_MELS else 1
        for i in range(N_MELS):
            s, e = i * group, min(i * group + group, 768)
            if e > s:
                proj[i, s:e] = 1.0 / (e - s)
        if group * N_MELS < 768:
            tail = 768 - group * N_MELS
            proj[:tail, group * N_MELS:768] += torch.eye(tail, device=proj.device)

        wmap = torch.einsum('cm,bmt->bct', proj, h_aligned)
        wmap = (wmap - wmap.mean()) / wmap.std().clamp(min=1e-6)

        mel_img   = mel.squeeze(0).squeeze(0).cpu()
        delta_img = delta.squeeze(0).squeeze(0).cpu()
        wmap_img  = wmap.squeeze(0).cpu()

        save_rgb_resized(mel_img, delta_img, wmap_img, out_path)

    except Exception as e:
        print(f"Error processing {fname}: {e}")
        continue

print("Completed and saved to:", output_folder)